# Delete Assets

In [4]:
main_folder = 'projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2'
skip_asset = ''

try:
    # List all assets immediately inside the main folder
    assets = ee.data.listAssets({'parent': main_folder}).get('assets', [])
    
    for asset in assets:
        asset_id = asset['id']
        asset_type = asset['type']
        
        # 1. Skip the specific asset you want to save
        if asset_id == skip_asset:
            print(f"Skipped (User Protected): {asset_id}")
            continue
            
        # 2. Skip sub-folders and image collections entirely (do not delete their contents)
        if asset_type in ['FOLDER', 'IMAGE_COLLECTION']:
            print(f"Skipped (Sub-folder/Collection ignored): {asset_id}")
            continue
            
        # 3. Delete individual files (Images, Tables, etc.) inside the main folder
        ee.data.deleteAsset(asset_id)
        print(f"Deleted asset: {asset_id}")
        time.sleep(0.1) # Brief pause to respect API rate limits

    print("\nBatch process finished successfully.")

except Exception as e:
    print(f"Error reading asset list: {e}")


Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2/gediAGB_nepal_608
Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2/gediAGB_nepal_609
Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2/gediAGB_nepal_61
Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2/gediAGB_nepal_610
Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2/gediAGB_nepal_611
Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2/gediAGB_nepal_612
Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2/gediAGB_nepal_613
Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2/gediAGB_nepal_614
Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2/gediAGB_nepal_615
Deleted asset: projects/ee-joshisur231/assets/nepal_agb_estimation/gedi_agb_points2

# Check Asset Size

In [ ]:
import ee 
ee.Initialize(project="ee-joshisur231")

In [89]:
import time
all_assets_data = []
def get_asset_metadata(asset_id):
    try:
        asset_metadata = ee.data.getAsset(asset_id)
        # Safely extract size; default to 0 if missing (like for folders/collections)
        size_bytes = int(asset_metadata.get('sizeBytes', 0))
        size_mb = size_bytes / (1024 * 1024)  # Accurate MB conversion (1024^2)
        all_assets_data.append(asset_metadata)
        return size_mb
    except Exception:
        return 0.0

def process_asset(asset_id, asset_type, depth=0):
    """Recursively processes and prints asset sizes based on folder structure."""
    indent = "  " * depth
    
    if asset_type in ['FOLDER', 'IMAGE_COLLECTION']:
        print(f"{indent}Entering: {asset_id} [{asset_type}]")
        try:
            # List all sub-assets inside this container
            sub_assets = ee.data.listAssets({'parent': asset_id}).get('assets', [])
            for sub_asset in sub_assets:
                # Recursively process the child asset
                process_asset(sub_asset['id'], sub_asset['type'], depth + 1)
                time.sleep(0.1) # Rate limiting pause for child assets
        except Exception as e:
            print(f"{indent}Error reading container {asset_id}: {e}")
    else:
        # It's a leaf node asset (IMAGE, TABLE, etc.)
        size_mb = get_asset_metadata(asset_id)
        print(f"{indent}--> {asset_id} | {asset_type} | Size: {size_mb:.2f} MB")

# Main execution block
main_folder = 'projects/ee-joshisur231/assets/'

try:
    print(f"Starting asset scan for: {main_folder}\n")
    # List top-level assets
    root_assets = ee.data.listAssets({'parent': main_folder}).get('assets', [])
    
    for asset in root_assets:
        process_asset(asset['id'], asset['type'], depth=0)
        time.sleep(0.1) # Rate limiting pause between top-level assets
        
except Exception as e:
    print(f"Error reading root asset list: {e}")

asset_data = pd.DataFrame(all_assets_data)

Starting asset scan for: projects/ee-joshisur231/assets/

--> projects/ee-joshisur231/assets/Guthichaur_test1 | TABLE | Size: 0.01 MB
Entering: projects/ee-joshisur231/assets/Rangeland_Degradation_Final [FOLDER]
  Entering: projects/ee-joshisur231/assets/Rangeland_Degradation_Final/final_results [FOLDER]
    --> projects/ee-joshisur231/assets/Rangeland_Degradation_Final/final_results/best_precTemp_ndvi_corr_monthID | IMAGE | Size: 0.94 MB
    --> projects/ee-joshisur231/assets/Rangeland_Degradation_Final/final_results/best_precTemp_ndvi_corr_monthID_era5 | IMAGE | Size: 6.14 MB
    --> projects/ee-joshisur231/assets/Rangeland_Degradation_Final/final_results/best_prec_ndvi_corr_monthID | IMAGE | Size: 0.58 MB
    --> projects/ee-joshisur231/assets/Rangeland_Degradation_Final/final_results/best_prec_ndvi_corr_monthID_era5 | IMAGE | Size: 4.14 MB
    --> projects/ee-joshisur231/assets/Rangeland_Degradation_Final/final_results/highest_correlation_preci | IMAGE | Size: 4.55 MB
    --> proje

In [92]:
asset_data["sizeMB"] = asset_data["sizeBytes"].astype(int)/ (1024 * 1024)

In [99]:
asset_data.sort_values(by="sizeMB", ascending=False)[["id", "sizeMB", "sizeBytes"]].to_csv("/home/proghead/Downloads/asset_details.csv")

In [1]:
for year in range(2000, 2025):
    if year in [2000, 2001, 2002]:
        continue
    print(year)

2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


In [3]:
import pandas as pd
import os

In [95]:
fp = r"/run/media/proghead/Nice!/work/Agricultural_Land_Abandonment_Revised/code/outputs/RF/results/with_landsat7/global/final/feature_importance"
files = os.listdir(fp)
feat_imp_list = []
for file in files:
    if "perm" not in file:
        year = file.split("_")[-1].split(".")[0]
        file_path = os.path.join(fp, file)
        df = pd.read_csv(file_path).sort_values("importance", ascending=False).drop(columns=['Unnamed: 0'])
        df["importance"] = df["importance"] * 100
        df["year"] = year
        feat_imp_list.append(df)
        print(
            f"Top 5 by mean importance in {year}:\n",
            f"\n{df.sort_values('importance', ascending=False).head(5)[["feature", "importance", "year"]]}\n"
        )

imp_df = pd.concat(feat_imp_list, ignore_index=True)

Top 5 by mean importance in 2000:
 
      feature  importance  year
18    tri_dtm   16.229138  2000
11    evi_p75   11.913797  2000
1   green_p25   11.517169  2000
14  msavi_p75   10.301028  2000
3     red_p25    7.434373  2000

Top 5 by mean importance in 2001:
 
      feature  importance  year
27    tri_dtm   19.567700  2001
1   green_p25   13.293853  2001
18    evi_p75    7.741485  2001
22  msavi_p75    7.170485  2001
4     red_p25    7.068050  2001

Top 5 by mean importance in 2002:
 
      feature  importance  year
21    tri_dtm    9.159863  2002
1   green_p25    8.381081  2002
3     red_p25    7.627949  2002
11   ndvi_p75    6.342300  2002
20      slope    6.322040  2002

Top 5 by mean importance in 2003:
 
      feature  importance  year
27    tri_dtm   13.372002  2003
1   green_p25   12.471680  2003
13   ndvi_p75   11.421336  2003
17    evi_p75   10.340275  2003
4     red_p25    8.770865  2003

Top 5 by mean importance in 2004:
 
      feature  importance  year
30    tri_dtm   

In [74]:
result_df = imp_df.groupby("feature").agg(
    importance_mean=("importance", "mean"),
    year_list=("year", list),
    year_count=("year", "count")
).reset_index()

In [98]:
print(
    "Top 20 by mean importance:\n",
    f"\n{result_df.sort_values('importance_mean', ascending=False).head(20)[["feature", "importance_mean", "year_count"]]}\n"
    # "\nTop 5 by count:\n",
    # f"\n{result_df.sort_values('year_count', ascending=False).head(20)[["feature", "importance_mean", "year_count"]]}\n"
)


Top 20 by mean importance:
 
             feature  importance_mean  year_count
47           tri_dtm        10.927916          25
3            evi_p75         7.672017          25
5          green_p25         7.448938          23
38           red_p25         6.855816          20
33          ndvi_p75         6.666122          25
10         msavi_p75         6.307455          25
46         swir2_p75         5.281569          24
40             slope         4.905782          25
14          ndvi_iqr         4.443350          22
37        red_median         4.172980          25
4       green_median         3.897712          25
13          ndmi_p75         3.807564          25
43         swir1_p75         3.423060          24
44      swir2_median         3.029919          23
39           red_p75         2.931817          20
42         swir1_p25         2.408311          21
1         evi_median         2.404448          20
7          msavi_iqr         2.381628          18
8       msavi_median 